# 03 - Retrieval: Ingredient Overlap & BM25

**Inputs:**
- `data/processed/cleaned_recipes.parquet` — output of notebooks 01 and 02 (must have `canonical_ingredients` column)
- `data/processed/alias_map.json` — Filipino ingredient alias map
- `ingredient_utils.py` — shared normalisation helpers

**Outputs:**
- Two retrieval functions: `overlap_retrieve()` and `bm25_retrieve()`
- Smoke-test results on 4 sample queries (English, Tagalog, Taglish, mixed)

Run cells in order. No GPU or internet needed.

## 0. Paths (works from `notebooks/` or the repo root)

In [1]:
import sys, os
from pathlib import Path

# Find the repo root by walking up from wherever the kernel started,
# instead of assuming the working folder is named "notebooks".
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "src" / "ingredient_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Could not find src/ingredient_utils.py - open this notebook from inside NLP-Mini-Project.")
os.chdir(ROOT)                          # so "data/processed/..." paths work
sys.path.insert(0, str(ROOT / "src"))   # so "import ingredient_utils" works
print("Working directory:", Path.cwd())


Working directory: C:\Users\Nina Claudia\Documents\college\fourth-year-first-sem\NLP\mini-project\NLP-Mini-Project


## 0b. Install dependencies (run once, then restart kernel if needed)

In [2]:
%pip install pandas pyarrow rank_bm25 rapidfuzz

Note: you may need to restart the kernel to use updated packages.


## 1. Load data and alias map

In [3]:
import json
import pandas as pd

# Bootstrapped import: works even if the paths cell above was skipped.
try:
    from ingredient_utils import load_alias_map, build_lookup, normalize_text
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    src = next(p for p in (here, *here.parents) if (p / "src" / "ingredient_utils.py").exists()) / "src"
    sys.path.insert(0, str(src))
    from ingredient_utils import load_alias_map, build_lookup, normalize_text

# ── recipes ──────────────────────────────────────────────────────────────────
df = pd.read_parquet("data/processed/cleaned_recipes.parquet")
print(f"Loaded {len(df)} recipes")
print("Columns:", df.columns.tolist())

# ── alias map ────────────────────────────────────────────────────────────────
alias_map = load_alias_map("data/processed/alias_map.json")
lookup, collisions = build_lookup(alias_map)
print(f"\nAlias map: {len(alias_map)} canonical ingredients, {len(lookup)} lookup entries")
if collisions:
    print("Collisions (review if many):", collisions[:5])

# ── verify canonical_ingredients column exists ───────────────────────────────
assert "canonical_ingredients" in df.columns, (
    "Missing 'canonical_ingredients' — run notebook 02 first and re-save the parquet."
)
print("\nSample canonical ingredients:")
df[["recipe_name", "canonical_ingredients"]].sample(3, random_state=42)


Loaded 1983 recipes
Columns: ['recipe_id', 'recipe_name', 'prep_time', 'cook_time', 'total_time', 'servings', 'ingredients', 'instructions', 'ingredient_names', 'ingredient_lines', 'ingredient_list', 'prep_time_min', 'cook_time_min', 'total_time_min', 'servings_min', 'servings_max', 'canonical_ingredients', 'unmapped_terms']

Alias map: 189 canonical ingredients, 452 lookup entries

Sample canonical ingredients:


,recipe_name,canonical_ingredients
582,Corned Beef Sinigang,"[water, beef, tamarind, fish sauce, salt, onio..."
1960,Mechado ala Alejandrino,"[onion, garlic, tomato, olive oil, bay leaf, b..."
1244,Ginisang Gulay At Tokwa,"[tofu, honey, chicken bouillon, soy sauce, ses..."


## 2. Query normalisation helper

Accepts a free-form ingredient list string (comma- or newline-separated, any language) and returns canonical ingredient names using the alias map.

In [4]:
import re

def parse_query(query: str, lookup: dict) -> list[str]:
    """Normalise a raw ingredient query string -> list of canonical ingredient names.

    Splits on commas, semicolons, newlines, and 'at'/'and' conjunctions,
    then runs each token through the alias map.
    Items that don't map to a canonical are kept as-is so they can still
    appear in BM25 token matching.
    """
    # split on common separators
    parts = re.split(r"[,;\n]|\bat\b|\band\b", query, flags=re.IGNORECASE)
    canonicals = []
    unmapped   = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        c, left = normalize_text(part, lookup)
        for x in c:
            if x not in canonicals:
                canonicals.append(x)
        unmapped.extend(left)
    return canonicals, unmapped


# quick sanity check
for q in [
    "toyo, bawang, sibuyas, manok",
    "pork, sitaw, tomatoes, fish sauce",
    "baboy at kamatis, patis, suka",
    "chicken, garlic, ginger, coconut milk, lemongrass",
]:
    c, u = parse_query(q, lookup)
    print(f"  Query : {q}")
    print(f"  Canon : {c}")
    if u:
        print(f"  Unmapped tokens: {u}")
    print()

  Query : toyo, bawang, sibuyas, manok
  Canon : ['soy sauce', 'garlic', 'onion', 'chicken']

  Query : pork, sitaw, tomatoes, fish sauce
  Canon : ['pork', 'string beans', 'tomato', 'fish sauce']

  Query : baboy at kamatis, patis, suka
  Canon : ['pork', 'tomato', 'fish sauce', 'vinegar']

  Query : chicken, garlic, ginger, coconut milk, lemongrass
  Canon : ['chicken', 'garlic', 'ginger', 'coconut milk', 'lemongrass']



## 3. Retriever A — Ingredient Overlap

**How it works:** counts how many of the query's canonical ingredients appear in a recipe's `canonical_ingredients` list, then ranks by that count (descending). Ties are broken by coverage — `matched / recipe_total` — so shorter recipes that fully match beat long ones with the same raw count.

No index to build; runs in O(n) over the recipe list.

In [5]:
def overlap_retrieve(
    query: str,
    df: pd.DataFrame,
    lookup: dict,
    top_k: int = 10,
) -> pd.DataFrame:
    """Rank recipes by canonical ingredient overlap with the query.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, matched_count, recipe_total,
        coverage, matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_set = set(query_canon)

    if not query_set:
        print("[overlap] Warning: query produced no canonical ingredients.")
        return pd.DataFrame()

    rows = []
    for _, recipe in df.iterrows():
        recipe_set = set(recipe["canonical_ingredients"])
        matched     = query_set & recipe_set
        matched_cnt = len(matched)
        recipe_total = max(1, len(recipe_set))
        coverage     = matched_cnt / recipe_total          # what fraction of the recipe you can fill
        rows.append({
            "recipe_id"          : recipe["recipe_id"],
            "recipe_name"        : recipe["recipe_name"],
            "matched_count"      : matched_cnt,
            "recipe_total"       : recipe_total,
            "coverage"           : round(coverage, 4),
            "matched_ingredients": sorted(matched),
            "query_canonicals"   : query_canon,
        })

    result = (
        pd.DataFrame(rows)
        .sort_values(["matched_count", "coverage"], ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    result.insert(0, "rank", result.index + 1)
    return result


print("overlap_retrieve() defined — ready for testing.")

overlap_retrieve() defined — ready for testing.


## 4. Retriever B — BM25

**How it works:** builds a BM25 index where each recipe's *document* is its list of canonical ingredient names (one token per ingredient). At query time the query's canonical names are used as the query tokens. BM25 rewards rare ingredients more than common ones (e.g. `lemongrass` matters more than `garlic`), which is a useful complement to raw overlap.

The index is built once and reused for all queries.

In [6]:
from rank_bm25 import BM25Okapi

def build_bm25_index(df: pd.DataFrame) -> BM25Okapi:
    """Build a BM25 index from canonical_ingredients.

    Each recipe is a 'document' whose tokens are its canonical ingredient names
    (multi-word names are kept as single tokens so 'coconut milk' stays together).
    """
    corpus = [
        # replace spaces within ingredient names with underscores so
        # 'coconut milk' becomes one BM25 token, not two
        [ing.replace(" ", "_") for ing in row]
        for row in df["canonical_ingredients"]
    ]
    return BM25Okapi(corpus)


bm25_index = build_bm25_index(df)
print(f"BM25 index built over {len(df)} recipes.")
print(f"Avg document length: {bm25_index.avgdl:.1f} tokens")

BM25 index built over 1983 recipes.
Avg document length: 11.0 tokens


In [7]:
import numpy as np

def bm25_retrieve(
    query: str,
    df: pd.DataFrame,
    bm25_index: BM25Okapi,
    lookup: dict,
    top_k: int = 10,
) -> pd.DataFrame:
    """Rank recipes with BM25 over canonical ingredient tokens.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, bm25_score,
        matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_tokens   = [ing.replace(" ", "_") for ing in query_canon]

    if not query_tokens:
        print("[bm25] Warning: query produced no canonical ingredients.")
        return pd.DataFrame()

    scores = bm25_index.get_scores(query_tokens)   # shape: (n_recipes,)

    top_indices = np.argsort(scores)[::-1][:top_k]

    rows = []
    query_set = set(query_canon)
    for rank, idx in enumerate(top_indices, start=1):
        recipe = df.iloc[idx]
        matched = sorted(query_set & set(recipe["canonical_ingredients"]))
        rows.append({
            "rank"               : rank,
            "recipe_id"          : recipe["recipe_id"],
            "recipe_name"        : recipe["recipe_name"],
            "bm25_score"         : round(float(scores[idx]), 4),
            "matched_ingredients": matched,
            "query_canonicals"   : query_canon,
        })

    return pd.DataFrame(rows)


print("bm25_retrieve() defined — ready for testing.")

bm25_retrieve() defined — ready for testing.


## 5. Test on sample queries

Four queries that cover English, Tagalog, Taglish, and a mixed multi-ingredient case.

In [8]:
SAMPLE_QUERIES = {
    "English"  : "chicken, garlic, onion, ginger, lemongrass, tamarind",
    "Tagalog"  : "baboy, bawang, sibuyas, suka, toyo, dahon ng laurel",
    "Taglish"  : "pork ribs, bawang, sibuyas, kamatis, patis",
    "Mixed"    : "chicken, luya, tanglad, coconut milk, patis, sili",
}

TOP_K = 5
DISPLAY_COLS_OVERLAP = ["rank", "recipe_name", "matched_count", "recipe_total", "coverage", "matched_ingredients"]
DISPLAY_COLS_BM25    = ["rank", "recipe_name", "bm25_score", "matched_ingredients"]

for label, query in SAMPLE_QUERIES.items():
    print("=" * 70)
    print(f"QUERY [{label}]: {query}")

    canon, unmapped = parse_query(query, lookup)
    print(f"  → canonicals : {canon}")
    if unmapped:
        print(f"  → unmapped   : {unmapped}")
    print()

    # ingredient overlap
    print("  [Ingredient Overlap — top 5]")
    ov = overlap_retrieve(query, df, lookup, top_k=TOP_K)
    if not ov.empty:
        print(ov[DISPLAY_COLS_OVERLAP].to_string(index=False))
    print()

    # BM25
    print("  [BM25 — top 5]")
    bm = bm25_retrieve(query, df, bm25_index, lookup, top_k=TOP_K)
    if not bm.empty:
        print(bm[DISPLAY_COLS_BM25].to_string(index=False))
    print()

QUERY [English]: chicken, garlic, onion, ginger, lemongrass, tamarind
  → canonicals : ['chicken', 'garlic', 'onion', 'ginger', 'lemongrass', 'tamarind']

  [Ingredient Overlap — top 5]
 rank                                     recipe_name  matched_count  recipe_total  coverage                                    matched_ingredients
    1            Lechon Manok with Grilled Corn Salad              6            16    0.3750 [chicken, garlic, ginger, lemongrass, onion, tamarind]
    2                    Sambal Chicken (Ayam Sambal)              6            22    0.2727 [chicken, garlic, ginger, lemongrass, onion, tamarind]
    3 Tinanglarang Manok (Chicken in Lemongrass Soup)              5             8    0.6250           [chicken, garlic, ginger, lemongrass, onion]
    4                                    Lechon Manok              5             9    0.5556           [chicken, garlic, ginger, lemongrass, onion]
    5                                 Chicken Binakol              5      

 rank                      recipe_name  matched_count  recipe_total  coverage                                 matched_ingredients
    1                Sprite Pork Adobo              6            11    0.5455 [bay leaf, garlic, onion, pork, soy sauce, vinegar]
    2                            Humba              6            11    0.5455 [bay leaf, garlic, onion, pork, soy sauce, vinegar]
    3                       Pork Adobo              6            11    0.5455 [bay leaf, garlic, onion, pork, soy sauce, vinegar]
    4             Boneless Crispy Pata              6            11    0.5455 [bay leaf, garlic, onion, pork, soy sauce, vinegar]
    5 Chili-Honey Glazed Lechon Kawali              6            11    0.5455 [bay leaf, garlic, onion, pork, soy sauce, vinegar]

  [BM25 — top 5]
 rank                      recipe_name  bm25_score                                 matched_ingredients
    1             Boneless Crispy Pata      6.2393 [bay leaf, garlic, onion, pork, soy sauce, vineg

 rank                                           recipe_name  matched_count  recipe_total  coverage                            matched_ingredients
    1                                      Pork Ribs Tinola              5            13    0.3846 [fish sauce, garlic, onion, pork_ribs, tomato]
    2                     Ginisang Pechay (Sauted Bok Choy)              4             7    0.5714            [fish sauce, garlic, onion, tomato]
    3                                      Ginisang Kamatis              4             7    0.5714            [fish sauce, garlic, onion, tomato]
    4                        Ginisang Okra with Fish Flakes              4             8    0.5000            [fish sauce, garlic, onion, tomato]
    5 Sarciadong Daing Recipe (Fried Dried Fish with Sauce)              4             8    0.5000            [fish sauce, garlic, onion, tomato]

  [BM25 — top 5]
 rank               recipe_name  bm25_score                            matched_ingredients
    1          

## 6. Side-by-side comparison on one query

Checks how much the two methods agree on the top-5 results.

In [9]:
COMPARE_QUERY = "baboy, bawang, sibuyas, suka, toyo, dahon ng laurel"
print(f"Query: {COMPARE_QUERY}\n")

ov_top = overlap_retrieve(COMPARE_QUERY, df, lookup, top_k=5)
bm_top = bm25_retrieve(COMPARE_QUERY, df, bm25_index, lookup, top_k=5)

overlap_names = set(ov_top["recipe_name"])
bm25_names    = set(bm_top["recipe_name"])

print("Overlap top-5 recipes:", list(ov_top["recipe_name"]))
print("BM25    top-5 recipes:", list(bm_top["recipe_name"]))
print()
print("In both :", sorted(overlap_names & bm25_names))
print("Only Overlap:", sorted(overlap_names - bm25_names))
print("Only BM25   :", sorted(bm25_names - overlap_names))

Query: baboy, bawang, sibuyas, suka, toyo, dahon ng laurel



Overlap top-5 recipes: ['Sprite Pork Adobo', 'Humba', 'Pork Adobo', 'Boneless Crispy Pata', 'Chili-Honey Glazed Lechon Kawali']
BM25    top-5 recipes: ['Boneless Crispy Pata', 'Chili-Honey Glazed Lechon Kawali', 'Sprite Pork Adobo', 'Humba', 'Pork Adobo']

In both : ['Boneless Crispy Pata', 'Chili-Honey Glazed Lechon Kawali', 'Humba', 'Pork Adobo', 'Sprite Pork Adobo']
Only Overlap: []
Only BM25   : []


## 7. Convenience wrapper — query from the command line

Call `retrieve(query)` to run both retrievers and print results in one go. This is handy for ad-hoc testing during the demo.

In [10]:
def retrieve(query: str, top_k: int = 10) -> dict:
    """Run both retrievers on a query and return a dict with both result DataFrames.

    Usage:
        results = retrieve("toyo, bawang, sibuyas, manok")
        display(results["overlap"])
        display(results["bm25"])
    """
    canon, unmapped = parse_query(query, lookup)
    print(f"Query : {query}")
    print(f"Canon : {canon}")
    if unmapped:
        print(f"Unmapped tokens (not in alias map): {unmapped}")
    print()

    ov = overlap_retrieve(query, df, lookup, top_k=top_k)
    bm = bm25_retrieve(query, df, bm25_index, lookup, top_k=top_k)

    print("─── Ingredient Overlap ──────────────────────────────")
    print(ov[["rank", "recipe_name", "matched_count", "recipe_total", "coverage"]].to_string(index=False))
    print()
    print("─── BM25 ────────────────────────────────────────────")
    print(bm[["rank", "recipe_name", "bm25_score"]].to_string(index=False))

    return {"overlap": ov, "bm25": bm}

results = retrieve("toyo, bawang, sibuyas, manok")

Query : toyo, bawang, sibuyas, manok


Canon : ['soy sauce', 'garlic', 'onion', 'chicken']

─── Ingredient Overlap ──────────────────────────────
 rank                                        recipe_name  matched_count  recipe_total  coverage
    1                        Homemade Chicken Longganisa              4             8    0.5000
    2                        Chicken Adobo with Tamarind              4             9    0.4444
    3                                     Chicken Bistek              4             9    0.4444
    4 Tamarind Chicken Adobo (Adobong Manok sa Sampalok)              4            10    0.4000
    5           Easy Pancit Canton with Chicken Barbecue              4            10    0.4000
    6                                     Chicken Bistek              4            11    0.3636
    7                          Instant Pot Chicken Adobo              4            11    0.3636
    8                                      Chicken Adobo              4            11    0.3636
    9           Chicken Pork

## 8. Notes

| | Ingredient Overlap | BM25 |
|---|---|---|
| **Scoring** | raw count + coverage ratio | IDF-weighted term frequency |
| **Rare-ingredient boost** | ✗ (garlic and lemongrass count the same) | ✓ (rarer = higher weight) |
| **Multi-word ingredients** | ✓ (canonical set, exact match) | ✓ (underscored token, exact match) |
| **Speed** | O(n) scan | O(n) dot-product, faster in practice |
| **Index needed?** | ✗ | ✓ (built once in cell 4) |

### How these connect to the rest of the RAG pipeline
- Both retrievers output a ranked list of `recipe_id`s. The RAG generator (notebook 04) will read the full recipe rows by those IDs to build the LLM context.
- Person 3's dense/hybrid retriever can be compared against these baselines using the same simulated pantry queries.
- For evaluation (Recall@k, MRR, nDCG@k), generate queries by masking 20–40 % of a known recipe's ingredients and check whether that recipe appears in top-k.

### Known limitations (for the paper/report)
- Neither method handles ingredient *quantities* or preparation style — `1 tsp garlic` and `1 head garlic` are treated the same.
- Unmapped Tagalog/Taglish terms fall through as raw tokens; extending `alias_map.json` improves recall.
- BM25 is sensitive to tokenisation: `"coconut milk"` → `"coconut_milk"` as one token prevents false matches with `"coconut water"`.